# ISS decoding with ISTDECO

This notebook uses the same SpaceTx formatting, registration, filtering, and channel-normalization path as the standard `ISS_decoding` workflow, then uses ISTDECO for joint spot detection and barcode decoding.

If SpaceTx data already exist, skip the formatting cell and start from the decoding section. Install the ISTDECO-enabled package with `python -m pip install -U "./ISS_decoding[istdeco]"`.

## We start by importing the necessary modules

In [ ]:
import ISS_decoding.SpaceTx_format as STX
import ISS_decoding.decoding as DEC
import pandas as pd
from pathlib import Path

## Format the images to SpaceTx format

The first thing to do before we can start the actual decoding is to transform our images (the resliced tiles) to the SpaceTx format.

To read more about the SpaceTx format, read the following: https://github.com/spacetx/sptx-format

### Parameters

`input_dir` = type: `str`. Path to the parent directory containing the preprocessed region folders (e.g., `/R1/`, `/R2/`, …).  
These region folders are automatically generated by the preprocessing module.  
The `input_dir` is referenced throughout this notebook.

`codebook_csv` = type: `str`. This is a file that associates a unique color sequence across ISS cycles to each gene. This file is a comma separated file with no header, in which the first column contains the gene name, while  columns 2 to 6 (in case of a 6 cycle experiment) contain numbers representing the expected positive DO_decorator in each cycle for that gene. 

`regions_to_process` = type:`list[int]` | None, default:None. A list of 1-based region indices defining which regions should be processed. If None → all detected regions are processed.

`output_dir_prefix` = type: `str` | None, default: `None`.  
Optional base directory where SpaceTx outputs should be written.

- If `output_dir_prefix` is `None`, SpaceTx outputs are written **inside each region directory under `input_dir`**, i.e.  
  `input_dir/R#/decoding/1_SpaceTX_format/`.

- If `output_dir_prefix` is set, SpaceTx outputs are written under:  
  `output_dir_prefix/R#/decoding/1_SpaceTX_format/`.

`pixel_to_um`  = type: `float`. 
Physical size of one pixel in microns (µm per pixel). This value determines the units of the spatial coordinates written to the experiment metadata and decoding outputs:

- `pixel_to_um = 1.0` (default) → coordinates are pixel-based.

- `pixel_to_um = 0.1625` (or microscope-specific value) → coordinates are in microns.


`channels` = type: `list`. The channels, in the order they were acquired in the microscope. Default = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"]

`DO_decorators` =  type: `list` **This point can be tricky to understand.** This shows how you associate the numbers in the codebook (ie 1,2,3,4) to a specific list of colors (DO_decorator) . In our lab, 1,2,3,4,5 correspond to ["AF750", "AF488", "Cy3", "Cy5", "At425"]. 

Default = ["AF750", "AF488", "Cy3", "Cy5", "At425"]. Users who follow our barcode design and readout schemes should not change this.

`nuclei_channel`  = type: `str`. This is the name of the channel that corresponds to your nuclei stained image. Default =  "DAPI".

`CARE` = type: `bool`.
If set to `True`, the function will use CARE-denoised retiled images as input.

- `CARE = False` (default):
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/`

- `CARE = True`:
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/CARE/`
This folder must already exist and should contain the outputs produced by the ISS_CARE preprocessing step.
This parameter does not run CARE itself. It only controls which image directory is used as input when building the experiment and codebook files.
Users should set `CARE = True` only after CARE denoising has been successfully completed for the corresponding regions and cycles.

In [ ]:
input_dir = '/path/to/regions/'
codebook_csv = '/path/to/codebook/'

In [ ]:
STX.make_spacetx_format(
    input_dir,
    codebook_csv,
    regions_to_process = None,    # or for example [2]
    output_dir_prefix = None,     # or '/path/to/preferred/output/dir'
    pixel_to_um = 1,
    channels = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"],
    DO_decorators = ["AF750", "AF488", "Cy3", "Cy5", "At425"],
    nuclei_channel = "DAPI",
    CARE = True                  # True if using CARE denoised images
    )


## Decode the SpaceTx formatted data

ISTDECO performs joint spot detection and decoding, so it does not use `spot_detection_mode`, `int_threshold`, or `sigma_vals`. `istdeco_quality` is a relative score rather than a calibrated probability; tune its threshold together with the intensity cutoff on representative data.

In [ ]:
DEC.process_experiment(
    input_dir,
    regions_to_process = None,             # or for example [1,4,5]
    output_dir_prefix = None,              # or '/path/to/preferred/output/dir'
    register = False,
    register_dapi = False,
    masking_radius = 15,
    normalization_method = 'MH',
    decode_mode = 'ISTDECO',
    dense = False,
    istdeco_kwargs = {
        'sigma': 1.2,
        'background': 1e-8,
        'scale': 1.0,
        'niter': 75,
        'acceleration': 1.0,
        'suppress_radius': 1,
        'tile_size': (512, 512),
        'overlap': None,
        'intensity_percentile': 99.0,
        'intensity_threshold': None,
        'quality_threshold': 0.5,
        'device': 'auto',
        'z_projection': 'max',
        'fake_barcode_fraction': 0.0, # e.g. 0.10 adds 10% unused negative-control barcodes
        'fake_barcode_seed': 0,       # reproducible controls across tiles and reruns
    },
    )

# Explore the decoded data

The decoding step writes a canonical region-level Parquet table plus a CSV compatibility copy. The main columns contain the decoded spot coordinates and target identity. Load one region at a time before applying method-specific quality filters.

#### Processing Reads for Individual Regions

We will now load one decoded region (`R1`, `R2`, etc.). The output directory for this workflow is `2_decoded_istdeco`.

In [ ]:
region = 'R1'

read_file = (
    Path(input_dir)
    / region
    / 'decoding'
    / '2_decoded_istdeco'
    / f"{region}_decoded_istdeco.parquet"
)

reads = pd.read_parquet(read_file)

The number of extracted raw reads for this region is:

In [ ]:
len(reads)